### This notebook computes a set of indicators from the WaterALLOC database for the 27 basins of IKI Project
- VSS9. Supply reliability for the agricultural sector" 
- VSB10. Availability of Water by Basin for the Agricultural Sector

**Spanish:** Confiabilidad de suministro: Porción de tiempo que la demanda de agua está completamente abastecida para el sector agrícola

**Created:** 1/22/2026 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 

**Status:** in progress

**QA Status:** reviewed by  
 
**Inputs:**   

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** The proportion of time during which water demand is fully met for the agrictulural sector.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 

#### Seleccion de rutas como funcion del usuario

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
#user = 'sbakar'
entry_folder= "" #"General\\"
user = 'etriana'

entry_path = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - {entry_folder}Interno"

db_path = fr"{entry_path}\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

wateralloc_db = fr"{entry_path}\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

# subbasins_shapefile = fr"{entry_path}\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"


In [3]:
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [4]:
conn = sqlite3.connect(db_path)

## NEED A NEW APPROACH/WAY TO FLAG FOR THIS

indicators_df = pd.read_sql_query(
    """
    SELECT IndID, TextID
    FROM Indicators
    WHERE SIG_Type = 'Wateralloc DB'
    """,
    conn
)

conn.close()

In [5]:
# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

wa_scenarios_df = pd.read_sql_query(
    """
    SELECT WaScnID, WaScnName
    FROM WaScenarios
    ORDER BY WaScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = wa_scenarios_df.loc[
    wa_scenarios_df['WaScnID'].isin([1, 3]), 'WaScnID'
].tolist()

# for all scenarios:
#scenario_ids = wa_scenarios_df['WaScnID'].tolist()

wa_scenarios_df

,WaScnID,WaScnName
0,1,CC_CMIP6_85_2050
1,3,Linea_Base_2020
2,4,Linea_Base_2020_Embalses


### Process all the WaterALLOC Scenarios
Los escenarios en WaterALLOC deben corresponder a un escenario en la base de datos de indicadores. Es esta seccion procesamos todos los escenarios en la base de datos vinculando con el escenario correspondiente en la base de datos de indicadores.

Este calculo necesita un indicador unico en la base de datos de indicadores para las combinaciones de  escenarios de indicadores y WaterALLOC.  
#### Method
Using SQL we attach the indicators database and execute an insert query in the indicators database using the processed data from the WaterALLOC scenarios database.  

Only the COMIDs processed to the WaterALLOC database are available.  The risk calculation should handle the missing COMID values.

#### Calculation Notes
The indicator for each COMID is queried as the .  

In [6]:
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)
cursor = conn_wa.cursor()
# Attach RiesgoDB database
attach_query = fr"ATTACH DATABASE '{db_path}' AS RiesgoDB;"
cursor.execute(attach_query)

In [7]:
## Create/update views that use existing tables to simplify indicator calculations
# View for Import/Export calculation
create_view_query = """
DROP VIEW IF EXISTS "main"."ImportExport fraction por COMID";
"""
cursor.execute(create_view_query)

create_view_query = """
CREATE VIEW "ImportExport fraction por COMID" AS SELECT a.RunID,a.Cuenca, a.COMID, 
	sum(importExport) AS NetImport,sum(importExport)/avg(TotAfluencia) AS ImportFract 
FROM (
	SELECT RunID,Cuenca,COMID, avg(Volumen) AS ImportExport
	FROM "WAMSS_Importacion y Exportacion anual por COMID"
	WHERE [Tipo] = 'Importacion'
	GROUP BY RunID,Cuenca, COMID
	UNION 
	SELECT RunID,Cuenca,COMID, avg(-Volumen) As ImportExport
	FROM "WAMSS_Importacion y Exportacion anual por COMID"
	WHERE [Tipo] = 'Exportacion'
	GROUP BY RunID,Cuenca, COMID
) as a
LEFT JOIN (
	SELECT RunID,Cuenca,avg(Afluencia) AS [TotAfluencia]
	FROM "WAMSS_Oferta anual por tipo por cuenca"
	GROUP BY RunID,Cuenca
) AS b ON b.RunID = a.RunID AND b.Cuenca = a.Cuenca 
GROUP BY a.RunID,a.Cuenca, a.COMID
"""
cursor.execute(create_view_query)

In [8]:
# Create temp table with ALL COMIDs from shapefile
subbasins_gdf.reset_index()[['COMID']].astype(int).to_sql(
    "AllSubbasinCOMIDs",
    conn_wa,
    if_exists="replace",
    index=False
)

pd.read_sql("SELECT * FROM AllSubbasinCOMIDs LIMIT 10", conn_wa)

,COMID
0,311153600
1,310832400
2,310825700
3,310282400
4,308754600
5,307736500
6,307078200
7,307183500
8,306538400
9,306599700


In [9]:
for waScn_ID in scenario_ids:

    print(f"\nProcessing WA scenario WaScnID={waScn_ID}")

    # Get RunIDs for this scenario
    run_ids_list = [row[0] for row in cursor.execute(
        f"SELECT RunID FROM WAMMS_RunsInfo WHERE WaScnID = {waScn_ID}"
    ).fetchall()]
    print(f"\tRunIDs for this scenario: {run_ids_list}")

    for ind_row in indicators_df.itertuples(index=False):
        indID = ind_row.IndID
        print(f"\tProcessing indicator {ind_row.TextID} (IndID={indID})")

        # -------------------------------
        # Indicator-specific logic
        # -------------------------------
        if ind_row.TextID == "VSS9":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[Confiabilidad]"
            and_where = "AND b.Sector = 'Agrario'"

        elif ind_row.TextID == "VSB10":
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_calc = "(b.[Oferta Local Sup] + b.[Oferta Entrada] - b.[Dem Local Sup])"
            and_where = ""

        elif ind_row.TextID == "VSS10":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[Confiabilidad]"
            and_where = "AND b.Sector = 'Poblacional'"

        elif ind_row.TextID == "VSS11":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[DeficitProm]"
            and_where = "AND b.Sector = 'Agrario'"

        elif ind_row.TextID == "VSS12":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[DeficitProm]"
            and_where = "AND b.Sector = 'Poblacional'"

        elif ind_row.TextID in ["VSS13_A", "VSS13_P"]:
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_calc = "b.[IndiceEstres_Sup]"
            and_where = ""

        elif ind_row.TextID in ["VCA16_A", "VCA16_P"]:
            source_table = "[WAMSS_Resiliencia por COMID y Sector]"
            value_calc = "b.[frequency_1_to_0]"
            and_where = (
                "AND b.Sector = 'Agrario'" if ind_row.TextID.endswith("_A") else "AND b.Sector = 'Poblacional'"
            )

        elif ind_row.TextID in ["VCA17_A", "VCA17_P", "VCA17_E"]:
            source_table = "[ImportExport fraction por COMID]"
            value_calc = "b.[ImportFract]"
            and_where = ""

        else:
            print(f"\t\tSkipping undefined indicator {ind_row.TextID}")
            continue

        # delete previous values for this scenario and indicator
        cursor.execute(f"""
            DELETE FROM [RiesgoDB].IndValues_WaALLOC
            WHERE WaScnID = {waScn_ID}
              AND IndID  = {indID};
        """)

        # check how many rows we have in the source table for this scenario and indicator
        count_source_rows = cursor.execute(f"""
            SELECT COUNT(*)
            FROM {source_table} AS b
            WHERE b.RunID IN ({','.join(map(str, run_ids_list))})
            {and_where};
        """).fetchone()[0]
        print(f"\t\tRows in source table for this indicator & scenario: {count_source_rows}")

        default_fill = 1 if ind_row.TextID in ["VSS9", "VSS10"] else 0

        # insert new calculated values
        insert_query = f"""
            INSERT INTO [RiesgoDB].IndValues_WaALLOC (WaScnID, IndID, COMID, Value)
            SELECT
                {waScn_ID} AS WaScnID,
                {indID} AS IndID,
                c.COMID,
                COALESCE(v.Value, {default_fill}) AS Value
            FROM AllSubbasinCOMIDs AS c
            LEFT JOIN (
                SELECT
                    b.COMID,
                    AVG({value_calc}) AS Value
                FROM {source_table} AS b
                WHERE b.RunID IN ({','.join(map(str, run_ids_list))})
                {and_where}
                GROUP BY b.COMID
            ) AS v
            ON v.COMID = c.COMID;
        """
        cursor.execute(insert_query)

        n_rows = cursor.execute(
            f"""
            SELECT COUNT(*)
            FROM [RiesgoDB].IndValues_WaALLOC
            WHERE WaScnID = {waScn_ID}
              AND IndID  = {indID};
            """
        ).fetchone()[0]

        print(f"\t\tInserted {n_rows} rows")

conn_wa.commit()


Processing WA scenario WaScnID=1
	RunIDs for this scenario: [1, 4, 6]
	Processing indicator VSB10 (IndID=310)
		Rows in source table for this indicator & scenario: 250236
		Inserted 3655 rows
	Processing indicator VCA16_A (IndID=5161)
		Rows in source table for this indicator & scenario: 186
		Inserted 3655 rows
	Processing indicator VCA16_P (IndID=5162)
		Rows in source table for this indicator & scenario: 59
		Inserted 3655 rows
	Processing indicator VCA17_A (IndID=5171)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VCA17_P (IndID=5172)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VCA17_E (IndID=5173)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VSS9 (IndID=409)
		Rows in source table for this indicator & scenario: 186
		Inserted 3655 rows
	Processing indicator VSS10 (IndID=410)
		Rows in source table for this indicator & sc

In [8]:
## Crear un loop para procesar cada escenario en el WaterALLOC DB 
# faster: iterate rows as namedtuples
for row in wa_scenarios_df.itertuples(index=True, name="Row"):
    waScn_ID = row.WaScnID
    print(f"\nProcessing WA scenario: {row.Scenario} (ScnID={row.WaScnID}) with indicator scenario name: {row.IndScnName}")
    
    scenario_name = scenarios_df.loc[
        scenarios_df['ScnName'] == row.IndScnName 
    ] 
    
    ind_ScnID = scenario_name.ScnID.values[0]
    print(f"\tLinked with scenario: {scenario_name.ScnName.values[0]} (ScnID={ind_ScnID})")
    
    # Delete previous entries for this indicator
    delete_query = fr"DELETE FROM [RiesgoDB].IndValues_WaALLOC WHERE WaScnID = {waScn_ID};"
    cursor.execute(delete_query)
    
    #loop through each indicator and get the IndID
    for ind_row in indicators_df.itertuples(index=True, name="IndRow"):
        indID = ind_row.IndID
        print(f"\t\tProcessing indicator ID: {indID} ({ind_row.TextID})")
        
        # Set source and value selection based on indicator TextID
        #   - Define source table
        #   - Define calculation or field to extract
        #   - Define any additional WHERE clauses
        ################################################################################    
        if ind_row.TextID == "VSS9":        
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_Calc = "[Confiabilidad]"
            and_where = " AND Sector = 'Agrario'"
        ################################################################################
        elif ind_row.TextID == "VSB10":
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_Calc = "avg(b.[Oferta Local Sup]+b.[Oferta Entrada]-b.[Dem Local Sup])"
            and_where = "" 
        ################################################################################
        elif ind_row.TextID == "VSS10":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_Calc = "[Confiabilidad]"
            and_where = " AND Sector = 'Poblacional'"
        ################################################################################
        elif ind_row.TextID == "VSS11":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_Calc = "[DeficitProm]"
            and_where = " AND Sector = 'Agrario'"
        ################################################################################
        elif ind_row.TextID == "VSS12":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_Calc = "[DeficitProm]"
            and_where = " AND Sector = 'Poblacional'"
        ################################################################################
        elif ind_row.TextID == "VSS13_A":
            #This indicator doesn't have a sector distinction
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_Calc = "avg(b.[IndiceEstres_Sup])"
            and_where = "" 
        ################################################################################
        elif ind_row.TextID == "VSS13_P":
            #This indicator doesn't have a sector distinction
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_Calc = "avg(b.[IndiceEstres_Sup])"
            and_where = "" 
        ################################################################################
        elif ind_row.TextID == "VCA16_A":
            source_table = "[WAMSS_Resiliencia por COMID y Sector]"
            value_Calc = "[frequency_1_to_0]"
            and_where = " AND Sector = 'Agrario'"
        ################################################################################
        elif ind_row.TextID == "VCA16_P":
            source_table = "[WAMSS_Resiliencia por COMID y Sector]"
            value_Calc = "[frequency_1_to_0]"
            and_where = " AND Sector = 'Poblacional'"
        ################################################################################
        elif ind_row.TextID == "VCA17_A":
            #Modelo Inputs/Outputs. Calcula el volumen annual promedio de importacion neta 
            # por COMID y lo divide entre la oferta annual promedio de la cuenca. 
            # Este indicador no se puede por sector
            source_table = "[ImportExport fraction por COMID]"
            value_Calc = "[ImportFract]"
            and_where = ""
        ################################################################################
        elif ind_row.TextID == "VCA17_P":
            #Modelo Inputs/Outputs. Calcula el volumen annual promedio de importacion neta 
            # por COMID y lo divide entre la oferta annual promedio de la cuenca. 
            # Este indicador no se puede por sector
            source_table = "[ImportExport fraction por COMID]"
            value_Calc = "[ImportFract]"
            and_where = ""
        ################################################################################
        elif ind_row.TextID == "VCA17_E":
            #Modelo Inputs/Outputs. Calcula el volumen annual promedio de importacion neta 
            # por COMID y lo divide entre la oferta annual promedio de la cuenca. 
            # Este indicador no se puede por sector
            source_table = "[ImportExport fraction por COMID]"
            value_Calc = "[ImportFract]"
            and_where = ""
        else:
            print(f"\t\t\tWarning: No source table or value calculation defined for indicator '{ind_row.TextID}'. Skipping.")
            continue
        ################################################################################    
        
        # Insert calculated values into IndValues_WaALLOC table
        query_avaltotal = fr"""
        INSERT INTO [RiesgoDB].IndValues_WaALLOC
            SELECT {waScn_ID} AS WaScnID, {indID} as IndID, b.comid AS COMID,
                {value_Calc} AS [Value] 
            FROM {source_table} AS b
            JOIN
                (SELECT RunID
                FROM WAMMS_RunsInfo
                Where WaScnID = {waScn_ID}) AS r 
                ON r.RunID = b.RunID
            WHERE comid not NULL
                {and_where}   
            GROUP BY b.comid;
        """
        # execute insert query and print the number of inserted rows
        rows_inserted = cursor.execute(query_avaltotal).rowcount
        print(f"\t\t\tInserted {rows_inserted} rows into IndValues_WaALLOC table.")
        
    
# Commit and close DB connection
conn_wa.commit()
conn_wa.close() 





Processing WA scenario: CC_CMIP6_85_2050 (ScnID=1) with indicator scenario name: CC CMIP6 85
	Linked with scenario: CC CMIP6 85 (ScnID=2)
		Processing indicator ID: 310 (VSB10)
			Inserted 543 rows into IndValues_WaALLOC table.
		Processing indicator ID: 409 (VSS9)
			Inserted 186 rows into IndValues_WaALLOC table.
		Processing indicator ID: 410 (VSS10)
			Inserted 59 rows into IndValues_WaALLOC table.
		Processing indicator ID: 411 (VSS11)
			Inserted 186 rows into IndValues_WaALLOC table.
		Processing indicator ID: 412 (VSS12)
			Inserted 59 rows into IndValues_WaALLOC table.
		Processing indicator ID: 4131 (VSS13_A)
			Inserted 543 rows into IndValues_WaALLOC table.
		Processing indicator ID: 4132 (VSS13_P)
			Inserted 543 rows into IndValues_WaALLOC table.
		Processing indicator ID: 5161 (VCA16_A)
			Inserted 186 rows into IndValues_WaALLOC table.
		Processing indicator ID: 5162 (VCA16_P)
			Inserted 59 rows into IndValues_WaALLOC table.
		Processing indicator ID: 5171 (VCA17_A)
	